# AutoARIMA - Automatic ARIMA Model Selection

This notebook demonstrates **automatic ARIMA model selection** using `forecastbox.auto.AutoARIMA`,
which implements the **Hyndman-Khandakar algorithm** (Hyndman & Khandakar, 2008).

## The Hyndman-Khandakar Algorithm

The algorithm automates the entire ARIMA model specification process:

1. **Determine differencing orders** $d$ and $D$:
   - Regular differencing $d$: selected via unit root tests (ADF and KPSS)
   - Seasonal differencing $D$: selected via seasonal strength test (OCSB test)
2. **Stepwise search** over $(p, q)$ and $(P, Q)$:
   - Start from a set of initial candidate models
   - Evaluate neighboring models by varying $p, q, P, Q$ by $\pm 1$
   - Rank by information criterion (AIC, AICc, or BIC)
   - Stop when no improvement is found
3. **Select the best model** with the lowest information criterion value

This is much faster than exhaustive grid search while still finding near-optimal models.

**Topics covered:**
- Loading and visualizing time series data
- Running AutoARIMA with diagnostic output
- Analyzing selected model parameters and residuals
- Forecasting with prediction intervals
- Applying AutoARIMA across multiple M3 series
- Tuning search parameters (stepwise vs exhaustive)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecastbox.auto import AutoARIMA

import sys
sys.path.insert(0, "..")
from utils.helpers import load_airline, load_m3_sample, get_series

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)

## 1. Loading Data

We use the classic **airline passengers** dataset (Box & Jenkins, 1976) — 144 monthly observations
of international airline passengers from 1949 to 1960. This series exhibits:

- **Upward trend**: passenger numbers grow over time
- **Multiplicative seasonality**: seasonal swings increase proportionally with the level
- **Non-stationarity**: requires differencing to stabilize

In [ ]:
# Load the airline dataset
df_airline = load_airline()
airline = df_airline["passengers"]

print(f"Series length: {len(airline)}")
print(f"Date range: {airline.index[0]} to {airline.index[-1]}")
print(f"Frequency: Monthly")

# Visualize the series
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(airline.index, airline.values, color="steelblue", linewidth=1.5)
ax.set_title("Airline Passengers (1949-1960)")
ax.set_xlabel("Date")
ax.set_ylabel("Passengers (thousands)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. How AutoARIMA Works

The Hyndman-Khandakar algorithm proceeds in two stages:

### Stage 1: Determine differencing orders

- **ADF test** (Augmented Dickey-Fuller): tests $H_0$: unit root exists. If $p$-value > 0.05, the series is non-stationary → increment $d$.
- **KPSS test**: tests $H_0$: series is stationary. If $p$-value < 0.05, reject stationarity → increment $d$.
- Both tests are applied iteratively until $d$ is determined (max $d = 2$).
- For seasonal data, **seasonal differencing** $D$ is determined similarly (max $D = 1$).

### Stage 2: Stepwise search over ARIMA orders

Starting from initial models (e.g., ARIMA(0,d,0), ARIMA(2,d,2), ARIMA(1,d,0)):
1. Fit each candidate and compute the information criterion (AIC, AICc, or BIC)
2. From the best model, try all neighbors: $p \pm 1$, $q \pm 1$, $P \pm 1$, $Q \pm 1$
3. If a neighbor improves the IC, adopt it and repeat step 2
4. Stop when no neighbor improves the IC

Let's run AutoARIMA with `trace=True` to see each step:

In [ ]:
# Run AutoARIMA with trace output to see the search process
auto_arima = AutoARIMA(
    seasonal=True,
    m=12,            # monthly data → seasonal period = 12
    stepwise=True,   # use stepwise search (faster)
    ic="aicc",       # corrected AIC (default, best for small samples)
    trace=True,      # print each model evaluated
)

result = auto_arima.fit(airline)

print(f"\nSelected model: ARIMA{result.order} x {result.seasonal_order}")
print(f"Information criterion ({result.ic_name}): {result.ic_value:.2f}")
print(f"Number of models evaluated: {result.n_fits}")
print(f"Search method: {result.search_method}")

## 3. Analyzing the Selected Model

Once AutoARIMA selects a model, we should analyze:

1. **Model summary**: parameter estimates, standard errors, significance
2. **Residual diagnostics**: are residuals white noise?
   - **ACF plot**: no significant autocorrelation remaining
   - **Ljung-Box test**: $H_0$: residuals are independent (want $p > 0.05$)

In [ ]:
# Model summary with parameter estimates
print(result.summary())

# Residual diagnostics
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf

residuals = result.model.resid

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residual time plot
axes[0].plot(residuals, color="steelblue", linewidth=0.8)
axes[0].axhline(y=0, color="red", linestyle="--", alpha=0.5)
axes[0].set_title("Residuals")
axes[0].set_ylabel("Residual")
axes[0].grid(True, alpha=0.3)

# ACF of residuals
plot_acf(residuals.dropna(), ax=axes[1], lags=24, alpha=0.05)
axes[1].set_title("ACF of Residuals")

# Histogram of residuals
axes[2].hist(residuals.dropna(), bins=20, color="steelblue", edgecolor="white", density=True)
axes[2].set_title("Residual Distribution")
axes[2].set_xlabel("Residual")

plt.tight_layout()
plt.show()

# Ljung-Box test for residual autocorrelation
lb_test = acorr_ljungbox(residuals.dropna(), lags=[12, 24], return_df=True)
print("\nLjung-Box Test:")
print(lb_test)
print("\nIf p-values > 0.05, residuals are consistent with white noise.")

## 4. Forecasting

AutoARIMA produces both **point forecasts** and **prediction intervals** at specified
confidence levels (default: 80% and 95%). Wider intervals indicate greater uncertainty.

We forecast 24 months ahead (2 years beyond the end of the data).

In [ ]:
# Forecast 24 months ahead
fc = result.forecast(h=24, level=(80, 95))
fc_df = fc.to_dataframe()

print("Forecast (first 6 months):")
print(fc_df.head(6).to_string())

# Plot historical data + forecast with prediction intervals
fig, ax = plt.subplots(figsize=(12, 5))

# Historical data
ax.plot(airline.index, airline.values, color="steelblue", linewidth=1.5, label="Historical")

# Create forecast index
last_date = airline.index[-1]
fc_index = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=24, freq="MS")

# Point forecast
ax.plot(fc_index, fc.point, color="darkorange", linewidth=2, label="Forecast")

# 95% prediction interval
if fc.lower_95 is not None and fc.upper_95 is not None:
    ax.fill_between(fc_index, fc.lower_95, fc.upper_95, alpha=0.15, color="darkorange", label="95% PI")

# 80% prediction interval
if fc.lower_80 is not None and fc.upper_80 is not None:
    ax.fill_between(fc_index, fc.lower_80, fc.upper_80, alpha=0.3, color="darkorange", label="80% PI")

ax.set_title(f"AutoARIMA Forecast — ARIMA{result.order} x {result.seasonal_order}")
ax.set_xlabel("Date")
ax.set_ylabel("Passengers (thousands)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. AutoARIMA on M3 Series

The M3 competition dataset contains time series with diverse characteristics. Let's apply
AutoARIMA to multiple monthly series and see which ARIMA specification is selected for each.

In [ ]:
# Load M3 sample dataset
m3 = load_m3_sample()
print("Available series:", m3["series_id"].unique().tolist())
print()

# Apply AutoARIMA to each monthly series
monthly_ids = m3[m3["frequency"] == "monthly"]["series_id"].unique()

results_table = []
for sid in monthly_ids:
    y = get_series(m3, sid)
    auto = AutoARIMA(seasonal=True, m=12, stepwise=True, ic="aicc")
    res = auto.fit(y)
    results_table.append({
        "Series": sid,
        "Order (p,d,q)": str(res.order),
        "Seasonal (P,D,Q,m)": str(res.seasonal_order),
        "AICc": round(res.ic_value, 2),
        "Models Evaluated": res.n_fits,
    })

results_df = pd.DataFrame(results_table)
print("AutoARIMA Results for Monthly M3 Series:")
print(results_df.to_string(index=False))

## 6. Tuning Parameters

Key parameters to control the AutoARIMA search:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `max_p` | 5 | Maximum AR order |
| `max_q` | 5 | Maximum MA order |
| `max_P` | 2 | Maximum seasonal AR order |
| `max_Q` | 2 | Maximum seasonal MA order |
| `ic` | `"aicc"` | Information criterion: `"aic"`, `"aicc"`, `"bic"` |
| `stepwise` | `True` | Stepwise search (fast) vs exhaustive grid search (slow but thorough) |

**Stepwise vs Exhaustive**: Stepwise search evaluates ~20-30 models; exhaustive can evaluate hundreds.
In practice, stepwise almost always finds the same model or one with a negligibly different IC value.

In [ ]:
import time

# Compare stepwise vs exhaustive search on airline data
# Stepwise search
t0 = time.time()
auto_stepwise = AutoARIMA(seasonal=True, m=12, stepwise=True, ic="aicc")
res_stepwise = auto_stepwise.fit(airline)
time_stepwise = time.time() - t0

# Exhaustive (grid) search
t0 = time.time()
auto_exhaustive = AutoARIMA(seasonal=True, m=12, stepwise=False, ic="aicc", max_p=3, max_q=3)
res_exhaustive = auto_exhaustive.fit(airline)
time_exhaustive = time.time() - t0

print("Stepwise Search:")
print(f"  Model: ARIMA{res_stepwise.order} x {res_stepwise.seasonal_order}")
print(f"  AICc: {res_stepwise.ic_value:.2f}")
print(f"  Models evaluated: {res_stepwise.n_fits}")
print(f"  Time: {time_stepwise:.2f}s")

print(f"\nExhaustive Search (max_p=3, max_q=3):")
print(f"  Model: ARIMA{res_exhaustive.order} x {res_exhaustive.seasonal_order}")
print(f"  AICc: {res_exhaustive.ic_value:.2f}")
print(f"  Models evaluated: {res_exhaustive.n_fits}")
print(f"  Time: {time_exhaustive:.2f}s")

print(f"\nSpeedup: {time_exhaustive / max(time_stepwise, 0.001):.1f}x")
print(f"AICc difference: {abs(res_stepwise.ic_value - res_exhaustive.ic_value):.2f}")

## Exercise 1: Apply AutoARIMA to monthly_trend_seasonal series — SOLUTION

Load the `monthly_trend_seasonal` series from the M3 sample dataset and fit an AutoARIMA model.
Report the selected order, plot the forecast for 12 periods ahead with prediction intervals.

In [ ]:
# Exercise 1 — SOLUTION: Apply AutoARIMA to monthly_trend_seasonal
# This series has both trend and seasonality, so we expect AutoARIMA to select
# a model with d=1 (for the trend) and seasonal differencing.

# Step 1: Load the series
y = get_series(m3, "monthly_trend_seasonal")
print(f"Series: monthly_trend_seasonal")
print(f"Length: {len(y)} observations")
print(f"Mean: {y.mean():.2f}, Std: {y.std():.2f}")

# Step 2: Fit AutoARIMA with seasonal period m=12
auto = AutoARIMA(seasonal=True, m=12, stepwise=True, ic="aicc", trace=True)
res = auto.fit(y)

print(f"\n--- Selected Model ---")
print(f"Order (p,d,q): {res.order}")
print(f"Seasonal (P,D,Q,m): {res.seasonal_order}")
print(f"AICc: {res.ic_value:.2f}")
print(f"Models evaluated: {res.n_fits}")

# Reference: Expected d=1 (trend requires differencing) and seasonal component present
print(f"\nDifferencing order d = {res.order[1]} (expected: 1 for trend)")
print(f"Seasonal differencing D = {res.seasonal_order[1]}")

# Step 3: Forecast h=12 with prediction intervals
h = 12
fc = res.forecast(h=h, level=(80, 95))

print(f"\nForecast (h={h}):")
print(fc.to_dataframe().to_string())

# Step 4: Plot historical + forecast with prediction intervals
fig, ax = plt.subplots(figsize=(12, 5))

# Historical data
ax.plot(range(len(y)), y.values, color="steelblue", linewidth=1.5, label="Historical")

# Point forecast
fc_x = range(len(y), len(y) + h)
ax.plot(fc_x, fc.point, color="darkorange", linewidth=2, label="Forecast")

# 95% prediction interval
if fc.lower_95 is not None and fc.upper_95 is not None:
    ax.fill_between(fc_x, fc.lower_95, fc.upper_95, alpha=0.15, color="darkorange", label="95% PI")

# 80% prediction interval
if fc.lower_80 is not None and fc.upper_80 is not None:
    ax.fill_between(fc_x, fc.lower_80, fc.upper_80, alpha=0.3, color="darkorange", label="80% PI")

ax.set_title(f"AutoARIMA Forecast — ARIMA{res.order} x {res.seasonal_order}\nmonthly_trend_seasonal (h={h})")
ax.set_xlabel("Period")
ax.set_ylabel("Value")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercise 2: Compare AIC vs BIC selection criteria — SOLUTION

Fit AutoARIMA on the airline dataset using `ic="aic"` and `ic="bic"`. Compare which model
each criterion selects. BIC penalizes model complexity more heavily — does it select a
more parsimonious (simpler) model?

In [ ]:
# Exercise 2 — SOLUTION: Compare AIC vs BIC selection criteria
# BIC penalizes complexity more heavily than AIC (penalty = log(n) vs 2),
# so it tends to select simpler (more parsimonious) models.

# Step 1: Fit AutoARIMA with AIC
auto_aic = AutoARIMA(seasonal=True, m=12, stepwise=True, ic="aic")
res_aic = auto_aic.fit(airline)

# Step 2: Fit AutoARIMA with BIC
auto_bic = AutoARIMA(seasonal=True, m=12, stepwise=True, ic="bic")
res_bic = auto_bic.fit(airline)

# Step 3: Build comparison table
aic_params = sum(res_aic.order) + sum(res_aic.seasonal_order[:3])
bic_params = sum(res_bic.order) + sum(res_bic.seasonal_order[:3])

comparison = pd.DataFrame({
    "Criterion": ["AIC", "BIC"],
    "Order (p,d,q)": [str(res_aic.order), str(res_bic.order)],
    "Seasonal (P,D,Q,m)": [str(res_aic.seasonal_order), str(res_bic.seasonal_order)],
    "IC Value": [round(res_aic.ic_value, 2), round(res_bic.ic_value, 2)],
    "Total AR/MA Params": [aic_params, bic_params],
    "Models Evaluated": [res_aic.n_fits, res_bic.n_fits],
})

print("AIC vs BIC Model Selection Comparison:")
print("=" * 75)
print(comparison.to_string(index=False))
print("=" * 75)

# Step 4: Analysis
print(f"\nAIC selected: ARIMA{res_aic.order} x {res_aic.seasonal_order} ({aic_params} AR/MA params)")
print(f"BIC selected: ARIMA{res_bic.order} x {res_bic.seasonal_order} ({bic_params} AR/MA params)")

if bic_params <= aic_params:
    print("\n→ As expected, BIC selects a more parsimonious (simpler) model.")
    print("  BIC's heavier penalty on complexity (log(n) vs 2 per parameter)")
    print("  discourages overfitting by favoring fewer parameters.")
else:
    print("\n→ In this case, both criteria selected similar complexity models.")
    print("  This can happen when additional parameters provide substantial fit improvement.")

# Step 5: Show top models from each criterion
print("\nTop 5 models by AIC:")
if res_aic.all_models is not None:
    print(res_aic.all_models.head(5).to_string())

print("\nTop 5 models by BIC:")
if res_bic.all_models is not None:
    print(res_bic.all_models.head(5).to_string())